The goal of this project is to build a classification model that predicts whether a customer will leave Beta Bank in the near future based on their historical behavior and account data. The goal is to have a F1 Score above 0.59. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from sklearn.utils import shuffle

In [2]:
# Loading the data
beta_data= pd.read_csv('/datasets/Churn.csv')

print(beta_data.shape)
print(beta_data.head())
print(beta_data.info())
print(beta_data.isnull())

(10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0     2.0       0.00              1          1               1   
1     1.0   83807.86              1          0               1   
2     8.0  159660.80              3          1               0   
3     1.0       0.00              2          0               0   
4     2.0  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4

Upon initial review of the data, I am noting that the data is primarily clean. No changes need to be made to any of the categorizations.  Noticing that almost all columns have full representation of values with the exception of tenure which is 909 missing values. This is something that should be addressed. I can see that there are several identifiers that will hold no bearing to the model itself including RowNumber, CustomerID, and Surname. Those will all be individual and unique to each seperate entry which cant be accounted for in prediction modeling. I am also noticing that the columns Geography and Gender are categorical and would require OHE (One Hot Encoding) to be able to manipulate the data properly in the ML model. 

In [3]:
#drop identifier columns with no predictive value
beta_data= beta_data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

#address missing tenure values with median value
beta_data['Tenure']= beta_data['Tenure'].fillna(beta_data['Tenure'].median())

#one hot encoding for categorical features

beta_data=pd.get_dummies(beta_data, columns=['Geography', 'Gender'], drop_first=True)

print(beta_data.shape)
print(beta_data.head())
print(beta_data.isnull().sum())

(10000, 12)
   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619   42     2.0       0.00              1          1   
1          608   41     1.0   83807.86              1          0   
2          502   42     8.0  159660.80              3          1   
3          699   39     1.0       0.00              2          0   
4          850   43     2.0  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geography_Germany  \
0               1        101348.88       1                  0   
1               1        112542.58       0                  0   
2               0        113931.57       1                  0   
3               0         93826.63       0                  0   
4               1         79084.10       0                  0   

   Geography_Spain  Gender_Male  
0                0            0  
1                1            0  
2                0            0  
3                0            0  
4                1

In [4]:
#check class balance
print('Class Balance:')
print(beta_data['Exited'].value_counts())
print()
print('Class Proportions:')
print(beta_data['Exited'].value_counts(normalize=True).round(3))


Class Balance:
0    7963
1    2037
Name: Exited, dtype: int64

Class Proportions:
0    0.796
1    0.204
Name: Exited, dtype: float64


In [5]:
#split into features and target
target=beta_data['Exited']
features=beta_data.drop('Exited', axis=1)

#split the data 3 ways for training, testing and validation
features_train, features_temp, target_train, target_temp, = train_test_split(
    features, target, test_size=.4, random_state=12345, stratify= target
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=.5, random_state=12345, stratify= target_temp)


print('Train size:', features_train.shape)
print('Validation size:', features_valid.shape)
print('Test Size:', features_test.shape)

Train size: (6000, 11)
Validation size: (2000, 11)
Test Size: (2000, 11)


when reviewing the class balance you can see that there is a significant imbalance between staying and leaving. This means a simple model will not work well as it could predict 0 every time and still get an 80% accuracy which would meet the standard threshold. We must be careful with this. 

In [6]:

#Numerical feature scaling
numerical_cols=['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
scaler= StandardScaler()
scaler.fit(features_train[numerical_cols])

features_train[numerical_cols]= scaler.transform(features_train[numerical_cols])
features_valid[numerical_cols]= scaler.transform(features_valid[numerical_cols])
features_test[numerical_cols]= scaler.transform(features_test[numerical_cols])

#Baseline Model with no imbalnce correction
model_baseline = LogisticRegression(random_state=12345, solver='liblinear')
model_baseline.fit(features_train, target_train)
predicted_valid= model_baseline.predict(features_valid)

print('Baseline Model no imbalance correction')
print('F1:', f1_score(target_valid, predicted_valid).round(3))
print('AUC-ROC:', roc_auc_score(target_valid, model_baseline.predict_proba(features_valid)[:,1]).round(3))


Baseline Model no imbalance correction
F1: 0.305
AUC-ROC: 0.787


/tmp/ipykernel_64/2849392528.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features_train[numerical_cols]= scaler.transform(features_train[numerical_cols])
/.venv/lib/python3.9/site-packages/pandas/core/indexing.py:1738: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value[:, i].tolist(), pi)
/tmp/ipykernel_64/2849392528.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

In [7]:
# class_weight balanced
best_f1 = 0
best_params = {}

for n in [100, 200]:
    for depth in [5, 7, 10]:
        model = RandomForestClassifier(
            n_estimators=n, 
            max_depth=depth, 
            class_weight='balanced', 
            random_state=12345
        )
        model.fit(features_train, target_train)
        predicted = model.predict(features_valid)
        f1 = f1_score(target_valid, predicted)
        print(f"n={n}, depth={depth} | F1: {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            best_params = {'n_estimators': n, 'max_depth': depth}

print("\nBest params:", best_params)
print("Best F1:", best_f1.round(3))

n=100, depth=5 | F1: 0.619
n=100, depth=7 | F1: 0.633
n=100, depth=10 | F1: 0.649
n=200, depth=5 | F1: 0.623
n=200, depth=7 | F1: 0.642
n=200, depth=10 | F1: 0.641

Best params: {'n_estimators': 100, 'max_depth': 10}
Best F1: 0.649


In [8]:
# Upsampling
def upsample(features, target, repeat):
    features_zeros= features[target==0]
    features_ones= features[target==1]
    target_zeros= target[target==0]
    target_ones= target[target==1]
    features_upsampled= pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled= pd.concat([target_zeros] + [target_ones] * repeat)
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345)
    return features_upsampled, target_upsampled

features_upsampled, target_upsampled= upsample(features_train, target_train, 4)

best_f1_up = 0
best_params_up= {}

for n in [100,200]:
    for depth in [5, 7, 10]:
        model= RandomForestClassifier(
            n_estimators=n,
            max_depth= depth,
            random_state=12345)
        model.fit(features_upsampled, target_upsampled)
        predicted=model.predict(features_valid)
        f1= f1_score(target_valid, predicted)
        print(f'n= {n}, depth= {depth}, F1: {f1:.3f}')
        if f1> best_f1_up:
            best_f1_up= f1
            best_params_up= {'n_estimators': n, 'max_depth': depth}
print('\nBest Params:', best_params_up)
print('Best F1:', best_f1_up.round(3))

n= 100, depth= 5, F1: 0.617
n= 100, depth= 7, F1: 0.635
n= 100, depth= 10, F1: 0.635
n= 200, depth= 5, F1: 0.620
n= 200, depth= 7, F1: 0.633
n= 200, depth= 10, F1: 0.645

Best Params: {'n_estimators': 200, 'max_depth': 10}
Best F1: 0.645


In [9]:
#DownSampling

def downsample(features, target, fraction):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]
    
    features_zeros = features_zeros.sample(frac=fraction, random_state=12345)
    target_zeros = target_zeros.sample(frac=fraction, random_state=12345)
    
    features_downsampled = pd.concat([features_zeros] + [features_ones])
    target_downsampled = pd.concat([target_zeros] + [target_ones])
    
    features_downsampled, target_downsampled = shuffle(
        features_downsampled, target_downsampled, random_state=12345
    )
    return features_downsampled, target_downsampled

features_downsampled, target_downsampled = downsample(features_train, target_train, 0.25)

best_f1_down = 0
best_params_down = {}

for n in [100, 200]:
    for depth in [5, 7, 10]:
        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=depth,
            random_state=12345
        )
        model.fit(features_downsampled, target_downsampled)
        predicted = model.predict(features_valid)
        f1 = f1_score(target_valid, predicted)
        print(f'n={n}, depth={depth} | F1: {f1:.3f}')
        if f1 > best_f1_down:
            best_f1_down = f1
            best_params_down = {'n_estimators': n, 'max_depth': depth}

print('\nBest Params:', best_params_down)
print('Best F1:', best_f1_down.round(3))

n=100, depth=5 | F1: 0.613
n=100, depth=7 | F1: 0.630
n=100, depth=10 | F1: 0.619
n=200, depth=5 | F1: 0.612
n=200, depth=7 | F1: 0.630
n=200, depth=10 | F1: 0.610

Best Params: {'n_estimators': 100, 'max_depth': 7}
Best F1: 0.63


In [10]:
# Decision Tree with Class_weight balanced

best_f1_dt = 0
best_params_dt = {}

for depth in [5, 7, 10, 15]:
    model_dt = DecisionTreeClassifier(
        max_depth=depth,
        class_weight='balanced',
        random_state=12345
    )
    model_dt.fit(features_train, target_train)
    predicted = model_dt.predict(features_valid)
    f1 = f1_score(target_valid, predicted)
    print(f'depth={depth} | F1: {f1:.3f}')
    if f1 > best_f1_dt:
        best_f1_dt = f1
        best_params_dt = {'max_depth': depth}

print('\nBest Params:', best_params_dt)
print('Best F1:', best_f1_dt.round(3))

depth=5 | F1: 0.555
depth=7 | F1: 0.572
depth=10 | F1: 0.535
depth=15 | F1: 0.524

Best Params: {'max_depth': 7}
Best F1: 0.572


When reviewing my 3 models I am seeing that the class_weight balanced model appears to work the best with a f1 score of .649 which falls above the .59 requirement. However I am not satisfied that this is the best this model can do. I intend to run threshold tuning before running my sanity test. 

In [11]:
#threshold tuning on best class_weight model
model_thresh= RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=12345
)

model_thresh.fit(features_train, target_train)
probabilities_valid= model_thresh.predict_proba(features_valid)[:,1]

best_f1_thresh= 0
best_threshold= .5

for threshold in np.arange(0.1, 0.9, 0.01):
    predicted= (probabilities_valid>= threshold).astype(int)
    f1= f1_score(target_valid, predicted)
    if f1 > best_f1_thresh:
        best_f1_thresh =f1
        best_threshold= threshold
print(f'Best Threshold: {best_threshold:.2f}')
print(f' Best F1 with Threshold tuning: {best_f1_thresh: .3f}')

Best Threshold: 0.47
 Best F1 with Threshold tuning:  0.656


as suspected the model was able to perform better while it is not a significant jump, it is enough to move forward with the tuned version. 

In [12]:
#Sanity Test- Class_weight balanced, and threshold tuning

final_model= RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=12345
)

final_model.fit(features_train, target_train)
probabilities_test= final_model.predict_proba(features_test)[:, 1]
predicted_test= (probabilities_test >= best_threshold).astype(int)

f1_test = f1_score(target_test, predicted_test)
auc_roc_test= roc_auc_score(target_test, probabilities_test)

print('Final Test Results')
print()
print(f'F1: {f1_test:.3f}')
print(f'Auc Roc: {auc_roc_test:.3f}')
print(f'Threshold Used: {best_threshold:.2f}')

Final Test Results

F1: 0.607
Auc Roc: 0.854
Threshold Used: 0.47


Sanity test holds. There was a slight decrease in the F1 performance however the performance is still higher than the threshold of.59. The AUC-ROC confirmed with a .854. 

Conclusion: 

I used 5 different approaches to handle the 80/20 class imbalance:

Baseline no correction (Logistic Regression): F1= .305
Class weight balanced (Decision Tree): F1= .572
Upsampling(repeat 4): F1= .645
Class Weight Balanced (Random Forest): F1= .649
Class Weight with Threshold Tuning(Random Forest): F1= .656


The best model was the Random Forest class weight balanced model with 100 estimators and a depth of 10 acheiving a F1= .607 and a AUC-ROC of .854 clearing the threshold of .59 required in the aformentioned sanity test. 